In [1]:
# ============================================================
# 13. INTEGRATE FORECAST + KNOWLEDGE BASE + LLM
# ============================================================

# Step 1: Import libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


# ============================================================
# Step 2: Load Actual Dataset
# ============================================================

file_path = "/content/smartagri_crop_yield_dataset.csv"

df = pd.read_csv(file_path)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())


# ============================================================
# Step 3: Prepare Data for Yield Forecasting
# ============================================================

features = [
    "Crop",
    "Soil_Type",
    "Soil_pH",
    "Nitrogen",
    "Phosphorus",
    "Potassium",
    "Temperature",
    "Rainfall",
    "Humidity"
]

target = "Yield"

X = df[features]
y = df[target]


# ============================================================
# Step 4: Encode Categorical Columns
# ============================================================

categorical_features = ["Crop", "Soil_Type"]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ],
    remainder="passthrough"
)


# ============================================================
# Step 5: Create Yield Forecasting Model
# ============================================================

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "regressor",
            RandomForestRegressor(
                n_estimators=100,
                random_state=42
            )
        )
    ]
)


# ============================================================
# Step 6: Split Dataset
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


# ============================================================
# Step 7: Train Model
# ============================================================

model.fit(X_train, y_train)

print("\nYield forecasting model trained successfully!")


# ============================================================
# Step 8: Check Forecast Accuracy
# ============================================================

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("\n--- Forecast Accuracy ---")
print("MAE  :", round(mae, 2))
print("RMSE :", round(rmse, 2))
print("R2   :", round(r2, 2))


# ============================================================
# Step 9: Build Agricultural Knowledge Base
# ============================================================

knowledge_base = df.groupby("Crop").agg({
    "Soil_Type": lambda x: x.mode()[0],
    "Soil_pH": "mean",
    "Nitrogen": "mean",
    "Phosphorus": "mean",
    "Potassium": "mean",
    "Temperature": "mean",
    "Rainfall": "mean",
    "Humidity": "mean",
    "Yield": "mean"
}).reset_index()

print("\n--- Agricultural Knowledge Base ---")
display(knowledge_base)


# ============================================================
# Step 10: Select One REAL Farm Case from Dataset
# ============================================================

farm_case = df.iloc[0]

print("\n--- Selected Farm Case ---")
print("Crop        :", farm_case["Crop"])
print("Soil Type   :", farm_case["Soil_Type"])
print("Soil pH     :", farm_case["Soil_pH"])
print("Nitrogen    :", farm_case["Nitrogen"])
print("Phosphorus  :", farm_case["Phosphorus"])
print("Potassium   :", farm_case["Potassium"])
print("Temperature :", farm_case["Temperature"])
print("Rainfall    :", farm_case["Rainfall"])
print("Humidity    :", farm_case["Humidity"])
print("Actual Yield:", farm_case["Yield"])


# ============================================================
# Step 11: Forecast Yield for the Farm
# ============================================================

farm_input = pd.DataFrame([{
    "Crop": farm_case["Crop"],
    "Soil_Type": farm_case["Soil_Type"],
    "Soil_pH": farm_case["Soil_pH"],
    "Nitrogen": farm_case["Nitrogen"],
    "Phosphorus": farm_case["Phosphorus"],
    "Potassium": farm_case["Potassium"],
    "Temperature": farm_case["Temperature"],
    "Rainfall": farm_case["Rainfall"],
    "Humidity": farm_case["Humidity"]
}])

predicted_yield = model.predict(farm_input)[0]

print("\n--- Yield Forecast ---")
print("Predicted Yield:", round(predicted_yield, 2))
print("Actual Yield   :", farm_case["Yield"])


# ============================================================
# Step 12: Determine Yield Status
# ============================================================

average_yield = df["Yield"].mean()

if predicted_yield >= average_yield * 1.10:
    yield_status = "High"
elif predicted_yield >= average_yield * 0.90:
    yield_status = "Medium"
else:
    yield_status = "Low"

print("\nYield Status:", yield_status)


# ============================================================
# Step 13: Retrieve Crop Knowledge
# ============================================================

crop_name = farm_case["Crop"]

crop_info = knowledge_base[
    knowledge_base["Crop"] == crop_name
]

print("\n--- Retrieved Crop Information ---")

if len(crop_info) > 0:

    crop_data = crop_info.iloc[0]

    print("Crop:", crop_data["Crop"])
    print("Average Soil Type:", crop_data["Soil_Type"])
    print("Average Soil pH:", round(crop_data["Soil_pH"], 2))
    print("Average Nitrogen:", round(crop_data["Nitrogen"], 2))
    print("Average Phosphorus:", round(crop_data["Phosphorus"], 2))
    print("Average Potassium:", round(crop_data["Potassium"], 2))
    print("Average Temperature:", round(crop_data["Temperature"], 2))
    print("Average Rainfall:", round(crop_data["Rainfall"], 2))
    print("Average Humidity:", round(crop_data["Humidity"], 2))
    print("Average Yield:", round(crop_data["Yield"], 2))

else:
    print("Crop information not available.")


# ============================================================
# Step 14: Create LLM Prompt
# ============================================================

llm_prompt = f"""
You are an agricultural advisory assistant.

Farmer's Crop:
{farm_case["Crop"]}

Farm Soil Type:
{farm_case["Soil_Type"]}

Farm Soil pH:
{farm_case["Soil_pH"]}

Nitrogen:
{farm_case["Nitrogen"]}

Phosphorus:
{farm_case["Phosphorus"]}

Potassium:
{farm_case["Potassium"]}

Temperature:
{farm_case["Temperature"]}

Rainfall:
{farm_case["Rainfall"]}

Humidity:
{farm_case["Humidity"]}

Predicted Yield:
{round(predicted_yield, 2)}

Yield Status:
{yield_status}

Agricultural Knowledge:
Average Soil pH = {round(crop_data["Soil_pH"], 2)}
Average Nitrogen = {round(crop_data["Nitrogen"], 2)}
Average Phosphorus = {round(crop_data["Phosphorus"], 2)}
Average Potassium = {round(crop_data["Potassium"], 2)}
Average Rainfall = {round(crop_data["Rainfall"], 2)}
Average Humidity = {round(crop_data["Humidity"], 2)}

Give the farmer simple and practical advice.

Include:
1. Yield condition
2. Soil recommendation
3. Water/irrigation recommendation
4. Fertilizer/nutrient recommendation
5. Possible risk
6. Simple action for the farmer

Do not invent information.
Use only the provided farm data and agricultural knowledge.
"""

print("\n================================================")
print("LLM-READY FARMER ADVISORY PROMPT")
print("================================================")

print(llm_prompt)


# ============================================================
# Step 15: Simple Advisory Without LLM API
# ============================================================

print("\n================================================")
print("FARMER ADVISORY")
print("================================================")

print("Crop:", crop_name)

if yield_status == "High":
    print("Yield condition: Expected yield is high.")
elif yield_status == "Medium":
    print("Yield condition: Expected yield is medium.")
else:
    print("Yield condition: Expected yield is low.")


# Soil recommendation
if farm_case["Soil_pH"] < 6:
    print("Soil: Soil pH is low. Consider suitable soil pH correction.")
elif farm_case["Soil_pH"] > 7.5:
    print("Soil: Soil pH is high. Suitable soil management may be required.")
else:
    print("Soil: Soil pH is within the normal range for this dataset.")


# Water recommendation
if farm_case["Rainfall"] < 500:
    print("Water: Rainfall is low. Additional irrigation may be required.")
elif farm_case["Rainfall"] > 1200:
    print("Water: Rainfall is high. Monitor excess water and drainage.")
else:
    print("Water: Rainfall condition is normal based on the selected threshold.")


# Nitrogen recommendation
if farm_case["Nitrogen"] < 50:
    print("Fertilizer: Nitrogen level is low. Check nitrogen requirement.")
else:
    print("Fertilizer: Nitrogen level does not show a low-level condition.")


# Humidity recommendation
if farm_case["Humidity"] > 80:
    print("Risk: High humidity may increase disease/fungal risk.")
else:
    print("Risk: No high-humidity risk detected by the selected rule.")


print("\n================================================")
print("INTEGRATION WORKFLOW")
print("================================================")

print("Farm Data")
print("    ↓")
print("Yield Forecasting Model")
print("    ↓")
print("Predicted Yield")
print("    ↓")
print("Agricultural Knowledge Base")
print("    ↓")
print("LLM Prompt")
print("    ↓")
print("Farmer Advisory")
print("    ↓")
print("Farmer Action")

Dataset loaded successfully!
Dataset shape: (600, 10)

Columns:
['Crop', 'Soil_Type', 'Soil_pH', 'Nitrogen', 'Phosphorus', 'Potassium', 'Temperature', 'Rainfall', 'Humidity', 'Yield']

First 5 rows:


,Crop,Soil_Type,Soil_pH,Nitrogen,Phosphorus,Potassium,Temperature,Rainfall,Humidity,Yield
0,Millets,Loamy,5.71,45.8,26.0,43.1,25.8,402.9,54.0,1.54
1,Sugarcane,Sandy,6.40,88.3,34.9,50.8,24.9,1213.0,88.3,55.75
2,Maize,Loamy,7.10,79.7,35.9,35.4,27.1,624.4,70.0,3.28
3,Millets,Loamy,6.10,29.3,29.8,23.0,22.7,466.9,48.7,1.29
4,Millets,Black,6.18,52.6,16.9,19.8,25.8,512.7,48.6,1.27



Yield forecasting model trained successfully!

--- Forecast Accuracy ---
MAE  : 1.03
RMSE : 2.47
R2   : 0.99

--- Agricultural Knowledge Base ---


,Crop,Soil_Type,Soil_pH,Nitrogen,Phosphorus,Potassium,Temperature,Rainfall,Humidity,Yield
0,Cotton,Loamy,6.832143,63.972619,28.198810,41.450000,27.188095,661.176190,59.428571,2.091667
1,Groundnut,Loamy,6.338375,48.747500,30.416250,38.233750,26.107500,605.927500,65.421250,1.801125
2,Maize,Alluvial,6.238837,76.087209,36.634884,39.570930,24.472093,687.873256,68.481395,3.639884
3,Millets,Loamy,6.127701,41.647126,23.457471,33.626437,26.927586,444.100000,57.987356,1.447471
4,Rice,Loamy,6.346500,81.408750,38.432500,41.823750,26.765000,1225.178750,80.870000,3.999375
5,Sugarcane,Loamy,6.996702,94.724468,41.947872,50.024468,27.767021,1439.805319,78.577660,66.285000
6,Wheat,Black,6.704607,69.113483,32.656180,37.677528,21.782022,536.641573,61.784270,3.322472



--- Selected Farm Case ---
Crop        : Millets
Soil Type   : Loamy
Soil pH     : 5.71
Nitrogen    : 45.8
Phosphorus  : 26.0
Potassium   : 43.1
Temperature : 25.8
Rainfall    : 402.9
Humidity    : 54.0
Actual Yield: 1.54

--- Yield Forecast ---
Predicted Yield: 1.54
Actual Yield   : 1.54

Yield Status: Low

--- Retrieved Crop Information ---
Crop: Millets
Average Soil Type: Loamy
Average Soil pH: 6.13
Average Nitrogen: 41.65
Average Phosphorus: 23.46
Average Potassium: 33.63
Average Temperature: 26.93
Average Rainfall: 444.1
Average Humidity: 57.99
Average Yield: 1.45

LLM-READY FARMER ADVISORY PROMPT

You are an agricultural advisory assistant.

Farmer's Crop:
Millets

Farm Soil Type:
Loamy

Farm Soil pH:
5.71

Nitrogen:
45.8

Phosphorus:
26.0

Potassium:
43.1

Temperature:
25.8

Rainfall:
402.9

Humidity:
54.0

Predicted Yield:
1.54

Yield Status:
Low

Agricultural Knowledge:
Average Soil pH = 6.13
Average Nitrogen = 41.65
Average Phosphorus = 23.46
Average Potassium = 33.63
Averag

In [ ]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=knowledge_base)

In [ ]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=knowledge_base)